# Gold Order Item Fact

This notebook builds the `fact_order_items` Gold model from the cleaned Silver order items dataset.

**Grain:** One row per `order_id` and `order_item_id`.

In [0]:
from pyspark.sql import functions as F

## 1. Define storage paths


In [0]:
SILVER_ORDER_ITEMS_PATH = (
    "abfss://silver@stnovacartdev.dfs.core.windows.net/"
    "olist/order_items"
)

GOLD_FACT_ORDER_ITEMS_PATH = (
    "abfss://gold@stnovacartdev.dfs.core.windows.net/"
    "olist/fact_order_items"
)

print(f"Silver source: {SILVER_ORDER_ITEMS_PATH}")
print(f"Gold target: {GOLD_FACT_ORDER_ITEMS_PATH}")

## 2. Read Silver order items

In [0]:
silver_order_items_df = (
    spark.read
    .format("delta")
    .load(SILVER_ORDER_ITEMS_PATH)
)

silver_order_item_count = silver_order_items_df.count()

print(f"Silver order item rows: {silver_order_item_count:,}")

display(silver_order_items_df.limit(10))

## 3. Validate required columns

In [0]:
required_columns = {
    "order_id",
    "order_item_id",
    "product_id",
    "seller_id",
    "shipping_limit_date",
    "price",
    "freight_value",
    "_silver_processed_at",
}

missing_columns = required_columns - set(silver_order_items_df.columns)

if missing_columns:
    raise ValueError(
        "Silver order items is missing required columns: "
        f"{sorted(missing_columns)}"
    )

print("Required column validation passed.")

## 4. Build order item fact

In [0]:
fact_order_items_df = (
    silver_order_items_df
    .select(
        "order_id",
        "order_item_id",
        "product_id",
        "seller_id",
        "shipping_limit_date",
        "price",
        "freight_value",
        "_silver_processed_at",
    )
    .withColumn(
        "item_total_value",
        F.col("price") + F.col("freight_value"),
    )
    .withColumn(
        "shipping_limit_date_key",
        F.date_format(
            F.to_date("shipping_limit_date"),
            "yyyyMMdd",
        ).cast("int"),
    )
    .withColumn("_gold_processed_at", F.current_timestamp())
)

display(fact_order_items_df.limit(10))

## 5. Validate order item fact

In [0]:
fact_order_item_count = fact_order_items_df.count()

duplicate_order_item_count = (
    fact_order_items_df
    .groupBy("order_id", "order_item_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

null_grain_count = (
    fact_order_items_df
    .filter(
        F.col("order_id").isNull()
        | F.col("order_item_id").isNull()
    )
    .count()
)

null_product_id_count = (
    fact_order_items_df
    .filter(F.col("product_id").isNull())
    .count()
)

null_seller_id_count = (
    fact_order_items_df
    .filter(F.col("seller_id").isNull())
    .count()
)

invalid_total_value_count = (
    fact_order_items_df
    .filter(
        F.col("item_total_value").isNull()
        | (F.col("item_total_value") < 0)
    )
    .count()
)

if fact_order_item_count == 0:
    raise ValueError("Order item fact is empty.")

if fact_order_item_count != silver_order_item_count:
    raise ValueError(
        "Order item fact row count does not match Silver order items. "
        f"Silver: {silver_order_item_count:,}, "
        f"Gold: {fact_order_item_count:,}"
    )

if duplicate_order_item_count > 0:
    raise ValueError(
        "Order item fact contains "
        f"{duplicate_order_item_count:,} duplicate grain combinations."
    )

if null_grain_count > 0:
    raise ValueError(
        "Order item fact contains "
        f"{null_grain_count:,} rows with null grain keys."
    )

if null_product_id_count > 0:
    raise ValueError(
        f"Order item fact contains {null_product_id_count:,} null product IDs."
    )

if null_seller_id_count > 0:
    raise ValueError(
        f"Order item fact contains {null_seller_id_count:,} null seller IDs."
    )

if invalid_total_value_count > 0:
    raise ValueError(
        "Order item fact contains "
        f"{invalid_total_value_count:,} invalid item total values."
    )

print(f"Order item fact rows: {fact_order_item_count:,}")
print("Order item fact grain validation passed.")

## 6. Write order item fact to Gold

In [0]:
(
    fact_order_items_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_FACT_ORDER_ITEMS_PATH)
)

print(f"Order item fact written to: {GOLD_FACT_ORDER_ITEMS_PATH}")

## 7. Validate Gold output

In [0]:
written_fact_order_items_df = (
    spark.read
    .format("delta")
    .load(GOLD_FACT_ORDER_ITEMS_PATH)
)

written_order_item_count = written_fact_order_items_df.count()

if written_order_item_count != fact_order_item_count:
    raise ValueError(
        "Gold order item fact write validation failed. "
        f"Expected: {fact_order_item_count:,}, "
        f"Written: {written_order_item_count:,}"
    )

print(f"Written order item fact rows: {written_order_item_count:,}")
print("Gold order item fact write validation passed.")

## 8. Inspect Gold order item fact

In [0]:
written_fact_order_items_df.printSchema()

display(
    written_fact_order_items_df
    .orderBy("order_id", "order_item_id")
    .limit(10)
)